SPDX-License-Identifier: AGPL-3.0-only
This team notebook is licensed under AGPL-3.0-only. See NOTICE.md; this does not license the app or datasets.


# RoadScan YOLO26s: road fine-tune on Kaggle (2x T4)

**Two runs, same data, keep the winner.** Commit this notebook twice: once with `RUN = "A"` (start from the main model, the
recipe that worked) and once with `RUN = "B"` (continue from the road model). Both must stay at or above RDD val mAP50-95
**0.346** (main 0.366 - 0.02), so B cannot quietly trade away more RDD accuracy. Compare the two `finetune_results.json`
tables on the road test (same test frames in both runs) and keep the better model.

**Before you run:**
1. **Upload 3 Kaggle datasets** (Datasets → New Dataset; Kaggle unzips them for you):
   - `RoadScan_models_and_data.zip`. After labeling new videos, re-zip it with the new folders inside
     `labeled_road_data/video_dataset/` and upload it as a **new version**.
   - `RoadScan_finetune_guide_and_code.zip`.
   - The `RDD_SPLIT` folder (11 GB) from `TRAINING_ROOT/RDD_SPLIT`, zipped.
2. **Set up the notebook:**
   - Add Input → add all 3 datasets.
   - Settings → Accelerator: **GPU T4 x2**.
   - Settings → Internet: **On** (needed for the pip install).
3. **Run the cells in order:**
   - Run cells 1–4 interactively, and check that cell 4 prints the expected counts.
   - Then use **Save Version → Save & Run All (Commit)**. That runs everything in the background (up to 12 h), so
     you can close the browser.
   - Download `road_v2_results.zip` from the version's **Output** tab.

Expected time (estimate, not measured on Kaggle): about 2–3 h of training, plus about 45 min scoring every
checkpoint.

**Do not change the training settings.** They are locked in `finetune_road.py`; see `README_FINETUNE.md`.


In [ ]:
# 1. Choose the run, then GPUs + exact Ultralytics version (the one the model was trained with; has cls_remap and cls_pw)
RUN = "A"   # "A" = start from the main model (recommended recipe) | "B" = continue from the fine-tuned road model
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q ultralytics==8.4.175
import ultralytics; assert ultralytics.__version__ == "8.4.175", ultralytics.__version__


In [ ]:
# 2. Find the 3 uploaded datasets, whatever you named them
from pathlib import Path
import shutil

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working/roadscan")

def find_one(pattern, what, ok=lambda p: True):
    hits = [p for p in INPUT.rglob(pattern) if "__MACOSX" not in p.parts and ok(p)]
    assert len(hits) == 1, f"{what}: expected 1 match for {pattern}, found {len(hits)}: {hits[:5]}"
    return hits[0]

# RDD under any folder name: the folder that has train/images and val/labels (not the video frames)
RDD = find_one("train", "RDD dataset", lambda p: (p / "images").is_dir() and (p.parent / "val" / "labels").is_dir()
               and "video_dataset" not in p.parts).parent.resolve()
CODE = find_one("finetune_road.py", "guide zip").parent
DATA = find_one("models", "package (models folder)", lambda p: (p / "1_original_yolo26s_1024.pt").exists()).parent
assert (RDD / "train" / "images").is_dir() and (RDD / "val" / "labels").is_dir(), f"bad RDD folder: {RDD}"
print("RDD :", RDD); print("CODE:", CODE); print("DATA:", DATA)


In [ ]:
# 3. Copy code + models + labeled frames to the writable folder (/kaggle/input is read-only). RDD stays where it is.
for src in (CODE, DATA):
    shutil.copytree(src, WORK, dirs_exist_ok=True)
print(sorted(p.name for p in WORK.iterdir()))


In [ ]:
# 4. Check the data only, no training. Expect: RDD train 26868 | video train N xK | video test M (M >= 100).
%cd /kaggle/working/roadscan
!python finetune_road.py --rdd "{RDD}" --check


In [ ]:
# 5. Train on both T4s: --batch 12 = 6 images per GPU, the same as the original run on one RTX 5070.
# Kaggle shows a 4-CPU machine, so 2 data workers per GPU. If multi-GPU fails, use: --device 0 --batch 6
# Both runs share one RDD floor: 0.346 = main 0.366 - 0.02. Run B starts at the road model (0.352), so its allowed drop is 0.006.
NAME, OUT, START, DROP = {
    "A": ("road_next_A", "new_road_A.pt", "models/1_original_yolo26s_1024.pt", 0.02),
    "B": ("road_next_B", "new_road_B.pt", "models/2_road_v1_yolo26s_1024_RECOMMENDED.pt", 0.006),
}[RUN]
!python finetune_road.py --rdd "{RDD}" --weights {START} --max-rdd-drop {DROP} --device 0,1 --batch 12 --workers 2 --out {OUT} --name {NAME}


In [ ]:
# 6. Results: row 0 = starting model. A new model is saved only if the road score rose and RDD dropped <= 0.02.
import json
res = json.loads(Path(f"runs/{NAME}/finetune_results.json").read_text())
print(f"{'checkpoint':28s} {'road mAP50':>10s} {'RDD mAP50-95':>13s}   road AP50 per class")
for r in res:
    pc = "  ".join(f"{k[:3]} {v:.2f}" for k, v in r["road"]["per_class_ap50"].items())
    print(f"{Path(r['checkpoint']).name:28s} {r['road']['map50']:10.3f} {r['rdd']['map50_95']:13.4f}   {pc}")
print("NEW MODEL SAVED" if Path(OUT).exists() else "No checkpoint passed; nothing saved")


In [ ]:
# 7. Pack the results for download (Output tab -> <NAME>_results.zip)
import zipfile
with zipfile.ZipFile(f"/kaggle/working/{NAME}_results.zip", "w") as z:
    for f in [OUT, f"runs/{NAME}/finetune_results.json", f"runs/{NAME}/results.csv",
              f"runs/{NAME}/args.yaml", f"lists/{NAME}/video_test.txt"]:
        if Path(f).exists():
            z.write(f, Path(f).name)
print(zipfile.ZipFile(f"/kaggle/working/{NAME}_results.zip").namelist())
